# 第14回 演習：コレスポンデンス分析（最終回）

## 前回からの接続

第13回では142か国を K-means で群れに分け、エルボーとシルエット係数でグループ数 K の根拠を測りました。あそこで効いていたのは「平均が計算できる数値データ」という前提です。今日の相手はジャンルと公開年代——選択肢の集まりなので、平均も距離もそのままでは定義できません。

それでも、数えることはできます。ジャンル×年代のマスに本数を入れたクロス表なら作れる。その表をカイ二乗で「関連があるか」まで確かめたうえで、一枚の地図に置き直すのが今日の対応分析です。数値データの地図を作り続けてきた最後に、カテゴリの地図を作ります。

## 今日の分析目標

**ジャンルと年代の結びつきを、一枚の地図で示したい。**

映画のジャンル×年代のクロス表を、カイ二乗で関連を確かめ、対応分析（CA）で一枚の地図にします。この演習では、その地図を自分の手で描いて読み解き、カイ二乗と CA がどうつながっているか、地図の距離を何と読むべきかまで踏み込みます。そして最終回として、全14回の解析フローを一枚に俯瞰します。TODOに取り組みながら、最後の「目標に答えられたか」で振り返りましょう。

## 学習ゴール

この回を終えると、次のことができるようになります。

- クロス表のカイ二乗が何のズレを測っているのかを、期待度数の作り方から説明できる
- カイ二乗の総量（総慣性 $=\chi^2/n$）を対応分析が軸ごとに切り分けている、という両者のつながりを言える
- 行（ジャンル）と列（年代）を重ねた地図を自分で描き、次元1が時代の軸になった理由を説明できる
- 行どうし・列どうしは距離で、行と列のあいだは原点から見た向きで読む、という読み分けができる
- 寄与率を添えて、その地図をどこまで信じて読んでよいかを言葉にできる


In [ ]:
# Colab には入っていないので、なければインストールする
import importlib.util
if importlib.util.find_spec('prince') is None:
    %pip install -q prince


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import prince

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
import japanize_matplotlib

plt.rcParams['font.size'] = 11
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
DATA_DIR = 'https://raw.githubusercontent.com/k0heiun0/applied_exercise/main/shared/data'   # データはこのリポジトリから読み込む
ct = pd.read_csv(f'{DATA_DIR}/movielens_genre_decade.csv', index_col=0)
ct.columns = ct.columns.astype(str)
print(ct)

### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は色の濃いところを扱います。


In [ ]:
# 図：分析の地図（全14回のどこにいるか）
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
here = {2, 3}
for i, (p, c, x) in enumerate(zip(phases, colors, [0.17, 0.5, 0.83]), start=1):
    on = i in here
    ax.text(x, 0.62, p, ha='center', va='center', fontsize=13 if on else 11,
            color='white', bbox=dict(boxstyle='round,pad=0.6', facecolor=c,
                                     alpha=0.95 if on else 0.25))
for x0, x1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(x1, 0.62), xytext=(x0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


## 1. カイ二乗で関連を確認

ジャンルと年代に関連があるかを、カイ二乗検定で確かめます。

### 深掘り：カイ二乗が測る「独立からのズレ」と、それが対応分析の土台になるまで

**主役の式**　カイ二乗統計量は、クロス表の「観測」と「無関係ならこうなるはず（期待）」のズレを、一つの数に集めたものです。

$$
\chi^2 \;=\; \sum_{i,j} \frac{(O_{ij}-E_{ij})^2}{E_{ij}},
\qquad E_{ij} \;=\; \frac{(\text{行 }i\text{ の合計})\times(\text{列 }j\text{ の合計})}{\text{総数}}
$$

記号をほどきます。$O_{ij}$ は実際にマスに入った本数（観測度数）、$E_{ij}$ は「ジャンルと年代がまったく無関係なら、このマスに入るはずの本数（期待度数）」です。期待度数は「行の合計 × 列の合計 ÷ 総数」で作ります——無関係なら、どのジャンルも各年代へ同じ割合で散らばるはずだから。分子 $(O_{ij}-E_{ij})^2$ は各マスのズレの二乗、それを期待度数 $E_{ij}$ で割って**相対的な大きさ**に直し、全マスで足し上げる。ズレの大きいマスが多いほど $\chi^2$ は大きく、「関連が強い」ことになります。観測度数の背後にある多項分布から出発して、この統計量が自由度 $(r-1)(c-1)$ のカイ二乗分布へ漸近することを示す分布論の導出は、この統計量を提案した Pearson の原論文（1900）と、漸近論を扱う数理統計学の成書に譲ります。

**このデータの実測値**　セットアップのセルが表示したクロス表は10ジャンル × 9年代（総本数 18,544 本）。これを次のセルで検定すると $\chi^2 \approx 548.4$、自由度は $(10-1)\times(9-1)=72$、p値は $\approx 2\times 10^{-74}$ という桁外れの小ささです。期待度数はどのマスも最小で約10本（5未満のマスはゼロ）なので、カイ二乗検定が前提とする「期待度数が小さすぎない」という条件も満たしています。**「時代とジャンルは無関係」という帰無仮説は、まず疑う余地なく棄却される**——時代が変わればジャンルの構成ははっきり偏る、と数字が断言しています。マスの本数が豊富な MovieLens を題材に選んだおかげで、この p値がここまで小さく、関連の有無で迷う余地がないのも心強いところです。

**カイ二乗と対応分析をつなぐ一本の糸：総慣性**　ここが今日の要です。カイ二乗は「関連が**ある**」ことは言えても「**どの**ジャンルが**どの**時代と結びつくか」は教えてくれません。その中身を地図にするのが対応分析（CA）ですが、両者は無関係な別物ではなく、**同じ $\chi^2$ を分解しているだけ**です。CA が地図に描く散らばりの総量（総慣性）は

$$
\text{総慣性} \;=\; \frac{\chi^2}{n} \;=\; \frac{548.4}{18544} \;\approx\; 0.0296
$$

と、**カイ二乗を総数 $n$ で割ったものにぴったり一致**します（後掲の「深掘りの数値確認」セルで確かめます）。CA は、この 0.0296 という「独立からのズレの総量」を、寄与の大きい軸から順に切り出して地図に置き直す作業なのです。つまり——**カイ二乗が「ズレの総額」を測り、CA が「その内訳」を地図にする**。この一本の糸を意識しながら、次の節で地図を描きます。

**なぜ期待度数で割るのか／自由度72はどこから来るのか**　二つ、細かいが大事な点を補います。まず $\sum (O-E)^2$ ではなく、**期待度数 $E$ で割ってから**足すのは、ズレを**相対化**するためです。総本数の多いマス（例：2000年代の Comedy）は、少しの偏りでも $(O-E)^2$ が大きく出ます。$E$ で割ることで「期待に対して何倍ズレたか」に直し、小さなマスの偏りも大きなマスの偏りも公平に足し合わせられます。次に自由度 $72$。行10・列9の表で、周辺合計（各行・各列の合計）を固定すると、自由に決められるマスは $(10-1)\times(9-1)=72$ 個ぶんです。この $72$ が「たまたまのズレでも $\chi^2$ がどのくらいまで膨らみうるか」の目安になり、実測の $548.4$ がそれを遥かに超えるからこそ、p値が極小になるのです。


In [ ]:
chi2, p, dof, _ = chi2_contingency(ct)
print(f'カイ二乗値: {chi2:.1f}, 自由度: {dof}, p値: {p:.2e}')
print('→ p値が非常に小さい＝関連は明確（時代でジャンル構成が偏る）')

### 図で追う：クロス表 → カイ二乗残差 → 地図

カイ二乗が「ズレの総額」を測り、対応分析が「その内訳」を地図にする——この関係を、手続きの流れとして一枚に並べておきます。左から順に、①本数を数えたクロス表、②各マスを期待度数からのズレに直した残差の表、③そのズレを分解して行と列を置き直した地図です。


In [ ]:
# 図：対応分析の3段階（クロス表 → カイ二乗残差 → 地図）
from matplotlib.patches import Rectangle

_chi2f, _pf, _doff, _Ef = chi2_contingency(ct)
_Rf = (ct.values - _Ef) / np.sqrt(_Ef)          # 標準化残差 (O-E)/sqrt(E)：90マス全体から計算
_caf = prince.CA(n_components=2, random_state=42).fit(ct)
_rf, _cf = _caf.row_coordinates(ct), _caf.column_coordinates(ct)
G = ['Romance', 'Comedy', 'Sci-Fi']             # 10ジャンルから3つ抜粋
D = ['1930', '1970', '2010']                    # 9年代から3つ抜粋
gi = [list(ct.index).index(g) for g in G]
di = [list(ct.columns).index(d) for d in D]
BLUE, RED = '#0066cc', '#e63946'

fig, axes = plt.subplots(1, 5, figsize=(14.5, 4.6),
                         gridspec_kw={'width_ratios': [1, 0.42, 1, 0.42, 1.3], 'wspace': 0.05})
fig.subplots_adjust(left=0.055, right=0.985, top=0.83, bottom=0.11)


def _table(ax, texts, faces, tcolors, title):
    for r in range(3):
        for c in range(3):
            ax.add_patch(Rectangle((c, 2 - r), 1, 1, facecolor=faces[r][c], edgecolor='#777', lw=1.0))
            ax.text(c + 0.5, 2 - r + 0.5, texts[r][c], ha='center', va='center',
                    fontsize=11, color=tcolors[r][c])
    for r, g in enumerate(G):
        ax.text(-0.12, 2 - r + 0.5, g, ha='right', va='center', fontsize=10)
    for c, d in enumerate(D):
        ax.text(c + 0.5, 3.10, f'{d}年代', ha='center', va='bottom', fontsize=10)
    ax.set_xlim(-1.55, 3.05); ax.set_ylim(-0.9, 3.8); ax.axis('off')
    ax.set_title(title, fontsize=11)


# ① クロス表（観測度数）
_table(axes[0], [[f'{ct.values[r, c]}' for c in di] for r in gi],
       [['#f2f2f2'] * 3 for _ in range(3)], [['black'] * 3 for _ in range(3)],
       '① クロス表：本数を数える\n（10ジャンル×9年代から3×3を抜粋）')
axes[0].text(0.75, -0.35, 'マスの数字＝その年代に公開された\nそのジャンルの本数（観測度数）',
             ha='center', va='top', fontsize=9, color='#444')

# ② カイ二乗残差
_cmap, _norm = plt.get_cmap('PuOr_r'), plt.Normalize(-6.5, 6.5)
_table(axes[2], [[f'{_Rf[r, c]:+.1f}' for c in di] for r in gi],
       [[_cmap(_norm(_Rf[r, c])) for c in di] for r in gi],
       [['white' if abs(_Rf[r, c]) > 3.5 else 'black' for c in di] for r in gi],
       '② カイ二乗残差：期待とのズレに直す\n$(O-E)/\\sqrt{E}$')
axes[2].text(0.75, -0.35, '濃い橙＝期待より多い／濃い紫＝期待より少ない',
             ha='center', va='top', fontsize=9, color='#444')

# ③ 対応分析の地図
ax = axes[4]
ax.scatter(_rf.iloc[:, 0], _rf.iloc[:, 1], c='#b9c7d6', s=45, zorder=2)
ax.scatter(_cf.iloc[:, 0], _cf.iloc[:, 1], c='#efc0c4', marker='^', s=45, zorder=2)
for g, off, ha in [('Romance', (8, 4), 'left'), ('Comedy', (8, -4), 'left'), ('Sci-Fi', (-2, 11), 'center')]:
    k = list(ct.index).index(g)
    x, y = _rf.iloc[k, 0], _rf.iloc[k, 1]
    ax.scatter([x], [y], c=BLUE, s=95, zorder=4)
    ax.annotate(g, (x, y), xytext=off, textcoords='offset points', color=BLUE, fontsize=10, ha=ha)
for d, off, ha in [('1930', (-8, 4), 'right'), ('1970', (8, 3), 'left'), ('2010', (-8, 6), 'right')]:
    k = list(ct.columns).index(d)
    x, y = _cf.iloc[k, 0], _cf.iloc[k, 1]
    ax.scatter([x], [y], marker='^', c=RED, s=95, zorder=4)
    ax.annotate(f'{d}年代', (x, y), xytext=off, textcoords='offset points', color=RED, fontsize=10, ha=ha)
ax.axhline(0, color='gray', lw=0.5, ls='--'); ax.axvline(0, color='gray', lw=0.5, ls='--')
ax.set_xlabel('次元1'); ax.set_ylabel('次元2')
ax.set_title('③ 対応分析の地図：残差を分解して置き直す\n青丸＝ジャンル、赤三角＝年代（灰色は残りの点）', fontsize=11)
ax.margins(0.20)

# ①→②→③ をつなぐ矢印
for k, lab, x0, x1 in [(1, '期待度数と\n比べる', 0.05, 0.85), (3, 'SVD で\n軸を取り出す', 0.00, 0.45)]:
    a = axes[k]; a.axis('off'); a.set_xlim(0, 1); a.set_ylim(0, 1)
    a.annotate('', xy=(x1, 0.50), xytext=(x0, 0.50),
               arrowprops=dict(arrowstyle='-|>', color='#555', lw=2, mutation_scale=20))
    a.text((x0 + x1) / 2, 0.55, lab, ha='center', va='bottom', fontsize=10, color='#333')
plt.show()


**この図の読み方**　①と②に並べたのは、10ジャンル×9年代のクロス表から Romance・Comedy・Sci-Fi の3行と1930・1970・2010年代の3列を抜き出した、**実データの一部**です（90マスすべてを並べると数字が潰れて読めないため。②の残差は90マス全体から計算した値をそのまま表示しています）。①のマスは本数そのもので、Sci-Fi の1930年代は3本、Comedy の2010年代は758本と、桁が二つ違います。この生の本数を「期待度数からどれだけズレたか」に直したのが②で、濃い橙が期待より多いマス、濃い紫が期待より少ないマスです。Romance×1930年代は $+6.0$ と表全体で最大のプラス、同じ Romance でも2010年代は $-5.7$ と大きなマイナス。Sci-Fi はちょうど裏返しで、1930年代 $-2.7$・2010年代 $+3.5$。Comedy は $+0.6$・$-2.2$・$+0.3$ と、どのマスも期待に近いところにとどまります。

③は、②のズレを軸に分解して行と列を置き直した地図です（青丸がジャンル、赤三角が年代、灰色はラベルを省いた残りの点）。②で読んだ関係が、そのまま位置に翻訳されています——1930年代（次元1で $+0.49$）と Romance（$+0.29$）は原点から同じ右方向へ離れており、これが②の $+6.0$ にあたります。逆に Sci-Fi（$-0.21$）は2010年代（$-0.16$）と同じ左側にいて、こちらは②の $+3.5$。ズレの小さい Comedy は、原点のすぐそば（原点からの距離 $0.03$）に落ち着きます。**本数をズレに直し、そのズレを分解して配置する**——対応分析の手続きは、この3枚で言い尽くせます。地図そのものは、2節で TODO① の座標から描きます。


### TODO①：対応分析を実行し、座標を取り出す

`prince.CA(n_components=2)` でクロス表 `ct` を分析し、行（ジャンル）と列（年代）の座標を取り出してください。

In [ ]:
# TODO: prince.CA で ct を分析し、row_coordinates(ct) と column_coordinates(ct) を取り出してください
# ヒント: ca = prince.CA(n_components=2, random_state=42).fit(ct)
# ヒント: rows = ca.row_coordinates(ct)、cols = ca.column_coordinates(ct)
...

## 2. 地図（バイプロット）を描く

行（ジャンル）と列（年代）を同じ地図に描きます。

### 深掘り：対応分析は「カテゴリのための PCA」——地図がどう組み上がり、次元1が「時代」になるのか

**CA は PCA の親戚**　第11・12回の PCA は、数値データの分散をいちばん多く残す軸を探して、点を低次元の地図に置き直しました。CA も発想はそっくりで、**扱う相手がクロス表になっただけ**です。手順を骨子だけ書くと——(1) 各マスを「観測 ÷ 期待 − 1」（独立からのズレ）に直し、期待度数で重みづけした**標準化残差**の行列を作る、(2) その行列を**特異値分解（SVD）**して、ズレの大きい方向から順に軸を取り出す、(3) ジャンル（行）と年代（列）を、その軸の座標として地図に置く。PCA が共分散行列を分解して「散らばりの大きい向き」を軸にしたのと同じ操作を、$\chi^2$ 残差に対して行っているのが CA です。だから PCA の直感——寄与率で信頼度を測る、上位2軸で要約する、バイプロットで行と列を重ねる——が、そっくりそのまま効きます。標準化残差の行列をどう組み立て、その特異値分解から行と列の主座標がどう定義されるのか——この行列表現の詳細は、対応分析だけを主題にした専門書（Greenacre『Correspondence Analysis in Practice』）の、行列による定式化を扱う章に譲ります。

**次元1が「時代の軸」になったのはなぜか**　地図を描くと、横軸（次元1）に沿って年代が古い順にきれいに並びます。TODO① で座標を取り出すと、年代の次元1は 1930年代 $+0.49$・1940年代 $+0.50$（右）から、2010年代 $-0.16$・1980年代 $-0.14$（左）へと単調に動きます。ジャンルも同じ軸に乗り、**右（古い）**に Romance $+0.29$・Drama $+0.13$、**左（新しい）**に Sci-Fi $-0.21$・Action $-0.19$・Horror $-0.14$。SVD は「いちばんズレを説明する方向」を第1軸に選ぶので、このデータで最大のズレ＝**時代によるジャンル構成の変化**が、そのまま次元1になったわけです。誰も「時代順に並べよ」とは指示していないのに、独立からのズレを分解しただけで時間軸が浮かび上がる——これが CA の醍醐味です。次元2（寄与 16.6%）は、同じ年代のなかでのジャンルの毛色の違い（Horror や Sci-Fi が下方向へ離れる、など）を拾う副次的な軸で、この二つで大枠が説明できます。

**発展：カテゴリが3つ以上なら多重対応分析（MCA）**　今日は「ジャンル × 年代」の2つのカテゴリでした。これが「ジャンル × 年代 × 制作国 × …」と3つ以上になると、CA の自然な拡張である**多重対応分析（MCA）**の出番です。各カテゴリをダミー変数（0/1）に展開した表に CA を当てる、と考えればよく、アンケートの多数の設問を一枚の地図に載せて「どの回答傾向どうしが近いか」を眺められます。考え方は CA と地続きなので、今日の地図の読み方が、そのまま MCA の入り口になります。

**別の見方：行と列は互いを説明し合う（双対）**　CA のもう一つの顔は、行の地図と列の地図が**表裏一体**だということです。各ジャンルの座標は「そのジャンルが多い年代たちの重心」に、各年代の座標は「その年代に多いジャンルたちの重心」に、それぞれ（軸ごとの特異値で伸縮したうえで）置かれます。ジャンルが年代を説明し、同時に年代がジャンルを説明する——この相互関係（双対性）を一枚の SVD が同時に解いているので、行と列を同じ地図に重ねられるのです。第12回のバイプロットで点（個体）と矢印（変数）を重ねられたのと、根っこは同じ発想です。

**順序カテゴリの癖：弧（アーチ効果）**　年代のように**順序のあるカテゴリ**を CA にかけると、点が直線ではなく**ゆるやかな弧**を描いて並ぶことがあります。次元1で「時代」という強い一次元の順序を捉えたあと、次元2が「両端（大昔と現代）と中間の違い」を拾うために生じる、CA で有名な**アーチ効果（Guttman 効果）**です。この地図の年代配置もきれいな直線には乗っておらず、その名残がうかがえます。弧はデータの欠陥ではなく強い順序性の副産物なので、次元2を無理に別の意味へ読みすぎないのが安全——この癖を知っておくと、順序尺度を CA・MCA にかけた図を落ち着いて読めます。

**「近い＝関連」をもう少し正確に**　スローガンとしての「近い点どうしは関連が強い」は、①行どうし・②列どうしについては（プロファイルが似ている、という意味で）正しく、直感的です。ただし次の節で見るとおり、③行と列のあいだでは、この一言をそのまま当てはめると足をすくわれます。地図を配ったときに「近いから関連が強いんですね」と早合点されがちなのが、まさに CA の落とし穴。だから「何と何の近さか」を必ず意識して読む——このひと呼吸が、CA を正しく使う鍵になります。


In [ ]:
ca = prince.CA(n_components=2, random_state=42).fit(ct)
rows = ca.row_coordinates(ct)
cols = ca.column_coordinates(ct)
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(rows.iloc[:,0], rows.iloc[:,1], c='#0066cc', s=90, zorder=3)
for i, n in enumerate(ct.index):
    ax.annotate(n, (rows.iloc[i,0], rows.iloc[i,1]), xytext=(6,4), textcoords='offset points', color='#0066cc', fontsize=9)
ax.scatter(cols.iloc[:,0], cols.iloc[:,1], c='#e63946', marker='^', s=90, zorder=3)
for i, n in enumerate(ct.columns):
    ax.annotate(f'{n}年代', (cols.iloc[i,0], cols.iloc[i,1]), xytext=(6,4), textcoords='offset points', color='#e63946', fontsize=9)
ax.axhline(0, color='gray', lw=0.5, ls='--'); ax.axvline(0, color='gray', lw=0.5, ls='--')
ax.set_xlabel('次元1'); ax.set_ylabel('次元2')
plt.tight_layout(); plt.show()

### TODO②：寄与率を確認する

この2次元が全体の何割を説明するか（寄与率）を確認してください。

### 深掘り：布置図の距離をどう読むか——行同士・列同士・行と列で、意味が違う（最大のつまずき）

ここが CA でいちばん間違えやすい所です。**同じ一枚の地図に描かれていても、「行と行」「列と列」「行と列」では、距離の意味がそれぞれ違います**。順に、実測値つきで整理します。

**① 行どうし（ジャンル×ジャンル）＝ カイ二乗距離**　二つのジャンルの点が近い ⇔ **年代ごとの本数配分（プロファイル）が似ている**。地図上のこの距離は、二つの行プロファイル間の**カイ二乗距離**に一致します（上位2軸へ落としたぶんだけの近似）。実測すると、Romance と Sci-Fi のカイ二乗距離は $0.53$ と全ペア中もっとも大きく（古い恋愛もの vs 新しい SF で時代配分がほぼ正反対）、Drama–Action は $0.34$。逆に Comedy–Drama は $0.19$ と小さく、どちらも「どの時代にも厚い」似た配分だと読めます。点の遠近を、そのままプロファイルの似て非なるさとして読める——これが行どうしの距離です。

**② 列どうし（年代×年代）＝ 同じくカイ二乗距離**　二つの年代が近い ⇔ **その年代のジャンル構成が似ている**。行とまったく同じ理屈で、列プロファイル間のカイ二乗距離になります。だから地図上で 1930年代と1940年代が隣り合うのは「ジャンル構成が似た隣接年代」だから、2010年代がそこから最も離れるのは「構成がいちばん違う」からです。

**③ 行と列（ジャンル×年代）＝ 単純な距離では読めない（classic mistake）**　ここが落とし穴です。ジャンルの点と年代の点の**生の直線距離**を、①②と同じ感覚で「近いから強く結びつく」と読むのは**誤り**です。対称マップでは、行と列はそれぞれ別に基準化されて同じ図に重ねてあるだけで、両者の間には「近い＝関連が強い」という単純な距離の意味が**そのままは成り立ちません**。正しくは、**原点から見た向き**と**原点からの遠さ**で読みます——あるジャンルと、ある年代が、**原点から同じ向きに、ともに遠く**離れていれば、その組は「独立より多い（引き合う）」。逆向きなら「独立より少ない（反発）」。数式で言えば、マスの「観測 ÷ 期待」比が、行と列の座標の内積で復元されます（$O_{ij}/E_{ij} = 1 + \sum_k F_{ik}G_{jk}/\sigma_k$、下の検証セルで確認）。実測すると Romance×1930年代は観測／期待 $=2.32$ と強い引き合い（原点から同じ右方向へともに遠い）、Romance×2010年代は $0.68$ と反発（互いに逆向き）。**行と列の関係は「距離」ではなく「向き」で読む**——これを取り違えないことが、CA を正しく使う分かれ目です。

**寄与率（＝各軸が説明する慣性の割合）を必ず添える**　TODO② の `eigenvalues_summary` は、この地図をどこまで信頼してよいかの通信簿です。総慣性 $0.0296$（＝$\chi^2/n$）のうち、次元1が $67.3\%$、次元2が $16.6\%$、合わせて **$83.9\%$（≒84%）** を2次元で説明します。①②③の距離の読みは本来すべて高次元での話ですが、8割超を2次元へ落とせているので、**この地図に限っては平面上の遠近・向きを安心して読める**——それが寄与率の意味です。もし寄与率が5割そこそこなら、平面の距離は当てにならず、第3軸以降も見なければなりません。

**原点＝平均的（時代を選ばない）**　原点は「全体平均のプロファイル」の位置です。だから原点に近いジャンルほど、どの時代にもまんべんなくある**定番**。実測の原点からの距離は Comedy $0.03$・Crime $0.06$ がもっとも中心寄りで、時代を選ばない屋台骨。対して Romance $0.29$・Sci-Fi $0.25$・Horror $0.21$・Action $0.21$ は原点から遠く、**特定の時代の色が濃い**ジャンルです。「中心にある＝重要でない」ではなく「時代普遍」と読むのが正解です。

**もう一歩：なぜ行と列を直接比べられないのか（基準化の流儀）**　行と列の距離を単純比較できないのは、両者が**別々のものさしで基準化**されているからです。今回の対称マップは行・列それぞれを主座標（principal coordinates）で置いたもので、行の広がりと列の広がりは同じスケールを共有していません。だからこそ③では「距離」でなく「原点からの向き」で読みました。用途によっては、片方を主座標・もう片方を標準座標に置く**非対称マップ**を使い、「各年代は、そこに多いジャンルの重心（バリセンター）に位置する」という読み（一方の点をもう一方の加重平均とみなす）を前面に出すこともあります。まず対称マップで向きを読み、必要なら非対称マップへ切り替える——それが実務での使い分けです。

**つまずき：本数の少ないマス・カテゴリに引きずられる**　CA は「独立からのズレ」を拾うため、**期待度数が小さいマスほど、わずかな本数の違いが大きなズレとして効いて**しまいます。極端に本数の少ないジャンルや年代が一つあるだけで、その点が地図の端へ大きく飛び、軸全体の向きを引っぱることさえあります。今回のデータは各マスが最小でも約10本と底が厚いので安定していますが、実務でスカスカのクロス表を扱うときは、まれなカテゴリを統合する・期待度数の小さいマスを警戒する、といった目配りが要ります。地図の端に一点だけ飛んでいる点を見たら、まず「それは本数が十分あっての位置か」を疑うのが安全です。


In [ ]:
# 深掘りの数値確認：カイ二乗と対応分析のつながり／布置図の距離の意味を、実値で確かめる
from scipy.stats import chi2_contingency
import prince
_ct = pd.read_csv(f'{DATA_DIR}/movielens_genre_decade.csv', index_col=0)
_ct.columns = _ct.columns.astype(str)
_N = _ct.values.astype(float); _n = _N.sum()
_chi2, _p, _dof, _E = chi2_contingency(_ct)
_ca = prince.CA(n_components=8, random_state=42).fit(_ct)
_F = _ca.row_coordinates(_ct).values; _G = _ca.column_coordinates(_ct).values
_sv = np.sqrt(np.array(_ca.eigenvalues_))
# (1) 総慣性 = χ²/n（カイ二乗と対応分析をつなぐ糸）
print(f'総慣性(全固有値の和)={np.array(_ca.eigenvalues_).sum():.4f}  chi2/n={_chi2/_n:.4f}  <- 一致')
# (2) 行どうしの地図距離 == 行プロファイルのカイ二乗距離
_P=_N/_n; _r=_P.sum(1); _c=_P.sum(0); _RP=_P/_r[:,None]
g=list(_ct.index)
_chi2d = lambda a,b: np.sqrt(np.sum((_RP[a]-_RP[b])**2/_c))
for a,b in [('Romance','Sci-Fi'),('Drama','Action'),('Comedy','Drama')]:
    ia,ib=g.index(a),g.index(b)
    print(f'{a:8s}-{b:8s}: カイ二乗距離={_chi2d(ia,ib):.3f}  全軸の地図距離={np.sqrt(((_F[ia]-_F[ib])**2).sum()):.3f}')
# (3) 行×列は「距離」でなく観測/期待で読む： O/E = 1 + sum_k F_ik G_jk / sigma_k
d=list(_ct.columns)
for gg,dd in [('Romance','1930'),('Romance','2010'),('Sci-Fi','2010')]:
    i,j=g.index(gg),d.index(dd)
    print(f'{gg:8s}x{dd}: 観測/期待={_P[i,j]/(_r[i]*_c[j]):.2f}  座標から復元={1+np.sum(_F[i]*_G[j]/_sv):.2f}')


In [ ]:
# TODO: ca.eigenvalues_summary を表示して、第1・第2次元の寄与率と累積を確認してください
# ヒント: print(ca.eigenvalues_summary)
...

### 総括：14回の解析フローを一枚の地図に——どの場面で、どの手法を選ぶか

最終回として、これまで歩いた道を一望します。分野は自転車・心臓病・生態・画像・自動車・国・映画とめぐりましたが、**貫く背骨は一つ**——「①目標を立てる → ②解析する → ③目標に立ち返る」の一周です。手法は、その②を埋める道具にすぎません。並べ直すと、全体はこう流れます。

**教師あり学習の一周（第1〜10回）**
- **汎化・過学習（第3回）**：モデルは訓練データを丸暗記するのでなく、未知に効く形にしたい。自由度を上げるとテスト誤差がいったん下がってやがて上がる U字を描く——すべての土台になる考え方。
- **前処理（第4回）**：標準化で単位をそろえ、カテゴリをダミー化する。距離や罰金を使う手法（正則化・クラスタリング・CA）は、ここが崩れると結果が全部ゆがむ。
- **回帰（第2回）と評価（第5回）**：問いを式（回帰モデル）にし、交差検証で「正しく測る」。テストは最後に、リークを防ぐ。
- **多重共線性 → 正則化（第6→7回）**：そっくりな変数どうしで係数が暴れる問題を、Ridge / Lasso の罰金で鎮め、係数を「読める」形にする。
- **分類とパイプライン（第8・9回）**：数値の予測から、クラス分けへ。前処理〜評価を一本のパイプラインに束ね、第10回の**総合演習**でフローを丸ごと1周した。

**教師なし学習へ（第11〜14回）**——正解ラベルがない世界。目的が「当てる」から「構造を見つける・要約する」へ変わる。
- **PCA（第11・12回）**：数値データの散らばりを最大に残す軸で、次元を圧縮し地図（バイプロット）にする。
- **クラスタリング（第13回）**：似たもの同士をグループに分ける。グループ数 K の根拠をエルボー・シルエットで測り、最後は目的で決めた。
- **対応分析（第14回・今日）**：カテゴリのクロス表を、カイ二乗の散らばりごと地図にして読む。いわばカテゴリ版の PCA。

**では、どの場面でどれを選ぶか**——手元のデータと問いから逆算する地図が、これです。
- **予測したい正解（ラベル・数値）があるか？**
  - **ある → 教師あり**。予測対象が**数値**なら**回帰**（第2〜7回）、**カテゴリ（分類先）**なら**分類**（第8・9回）。似た変数が多い・係数を安定させたいなら**正則化**を重ねる。
  - **ない → 教師なし**。何をしたいかで分岐する：
    - **変数を減らして可視化・要約したい（数値データ）→ PCA／バイプロット**。
    - **個体をグループに分けたい → クラスタリング**（丸い塊なら K-means、非球状なら DBSCAN、構造をじっくり眺めたいなら階層クラスタリング）。
    - **カテゴリ×カテゴリの結びつきを見たい → 対応分析（CA）**、カテゴリが3つ以上なら**MCA**。
- どの枝を選んでも共通するのは、**前処理（第4回）で土俵をそろえ、交差検証や寄与率・シルエットで「どこまで信じてよいか」を測り、関連と因果を混同しない**こと。手法は違っても、確かめる作法は同じです。

**変わらなかった四つの心構え**　手法が変わっても、これだけは全14回で一度も変わりませんでした。**(1)** 目標から始め、目標に立ち返る。**(2)** 正しく測る（テストは最後・リーク防止・指標は複数で）。**(3)** 手を動かして、仕組みを自分で確かめる。**(4)** 当てるだけでなく「なぜ」を語り、限界も正直に書く。道具はいつか古びても、この姿勢は古びません。14回、お疲れさまでした——学んだこの地図を手に、次はあなた自身のデータへ。

**一枚の早見表**

| 問い | データ | 手法（学んだ回） |
|---|---|---|
| 数値を予測したい | 数値の説明変数＋数値の正解 | 回帰・正則化（第2〜7回） |
| クラスを当てたい | 説明変数＋カテゴリの正解 | 分類・パイプライン（第8〜10回） |
| 変数を要約・可視化したい | 数値、正解なし | PCA・バイプロット（第11・12回） |
| 個体をグループ分けしたい | 数値、正解なし | クラスタリング（第13回） |
| カテゴリの結びつきを見たい | クロス表、正解なし | 対応分析・MCA（第14回） |

表は出発点にすぎません。実際のデータは「数値とカテゴリが混在」「正解が一部だけある」など、枠にきれいには収まらないことも多い。そのときこそ、②を埋める道具を**組み合わせます**——前処理で整えてから回帰、PCA で減らしてからクラスタリング、といった具合に。14回で身につけたのは、個々の手法というより、この**地図の上を目的に沿って歩く力**です。手法は道具、主役はいつも「何を知りたいか」という問いのほう。迷ったら①に戻り、「自分はいま何に答えようとしているのか」を書き出すところから始めてください。

**地図の外にも道は続く**　この14回で歩いたのは、統計的なデータ分析の王道の一区画です。地図の外には、まだ広い世界があります——正解が一部にしか付いていない**半教師あり学習**、試行錯誤で方策を学ぶ**強化学習**、順序と依存を扱う**時系列**、そして画像や言葉を相手にする**深層学習**。どれも今回は踏み込みませんでしたが、土台は共通です。「目標を立て、正しく測り、手を動かし、限界を語る」——この四つの心構えと、前処理・評価・可視化という基本の作法は、どの新しい手法を学ぶときも、そのまま最初の足場になります。新しい道具に出会ったら、まず「これは①〜③のどこを、どう埋める道具か」と問うてみてください。見慣れた地図の上に、ちゃんと居場所が見つかるはずです。


## 目標に答えられたか

- 今日の目標は「ジャンルと年代の結びつきを、一枚の地図で示したい」でした
- TODO①②を経て描いた地図で、横軸（次元1）は何を表していましたか？（古い年代と新しい年代はどちら側？）
- 古い時代の近くにあるジャンルは？ 新しい時代の近くにあるジャンルは？
- 原点近くのジャンル（時代を選ばない定番）は何でしたか？
- 寄与率は何%でしたか？ この地図はどのくらい信頼して読めますか？
- 14回、お疲れさまでした！ 学んだ手法のうち、自分のデータに使ってみたいものは？

## 今日の要点

- カテゴリのデータには平均も距離もそのままでは使えない。だから、まず数えてクロス表にするところから話を始める
- カイ二乗が答えるのは「関連があるか」までで、どの組み合わせが強いかは教えてくれない。中身を知りたければズレを分解するしかない
- その分解が対応分析。総慣性 $\chi^2/n = 0.0296$ を軸ごとに切り分けているだけで、カイ二乗と別ものの手法ではない
- 次元1が時代の軸になったのは、そう指示したからではなく、独立からのズレが最大になる方向がたまたま時代だったから
- 行どうし・列どうしの近さは、プロファイルの似かた（カイ二乗距離）として距離で読んでよい
- 行と列のあいだだけは距離で読めない。原点から同じ向きに、ともに遠いかどうかで引き合い・反発を読む
- 寄与率を添えないと地図は読めない。83.9% を2次元で説明できているから平面の遠近を信じてよい、と言えるのが寄与率の役目


## 14回を終えて

14回、お疲れさまでした。自転車・心臓病・生態・画像・自動車・国・映画と分野を渡り歩きましたが、やっていたことは毎回同じでした——①目標を立て、②手を動かして解析し、③結果を目標に照らす。第10回の総合演習で通してたどり、今日もう一度たどったこのフローを、これからは自分の手で回していけます。

手に入れた道具立ては、回帰と正則化、分類とパイプライン、次元縮約（PCA・バイプロット）、クラスタリング、そして今日の対応分析。どれを使うときも、探索と前処理で土俵をそろえ、汎化を意識して交差検証で正しく測る、という作法は変わりませんでした。数値にもカテゴリにも、正解ラベルのあるデータにもないデータにも、それぞれ入口があります。あとは、あなた自身の問いを①に置くところから始めてください。


## 課題（提出）

**提出するもの**: 応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

2節の地図は、1930年代から2010年代までの全9年代で描いたものでした。今度は年代を **1990年代以降の3つ（1990・2000・2010）** に絞って、同じ分析をやり直します。

クロス表 `ct` からこの3列だけを取り出したものを `ct_new` と名づけ、`prince.CA(n_components=2, random_state=42)` で分析し直してください。そして 2節のコードの `ct` を `ct_new` に（`ca` を `ca_new` に）置き換えて、同じ形の地図（ジャンルと年代のバイプロット）を描き、`eigenvalues_summary` で寄与率を表示してください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`ct` の列名は `'1930'` のような**文字列**です。`ct_new = ct[['1990', '2000', '2010']]` または `ct.loc[:, '1990':]` で3列を取り出せます。

</details>


### 応用②（判断）

応用①で絞ったあとの分析で、**第1次元の寄与率は何%**ですか。**小数第1位まで**（例: 12.3）で答えてください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`eigenvalues_summary` の `% of variance` 列は `'12.34%'` のような文字列なので、そのままでは丸められません。
数値がほしいときは `ca_new.percentage_of_variance_[0]` を使い、`round(..., 1)` で丸めます。

</details>


### 応用③（解釈）

2節の地図（全9年代）と応用①の地図（1990年代以降）を見比べてください。最も変わった点は何ですか。
**映画配信サービスの企画担当者**に向けて、**近づいた・離れたジャンルと年代**に触れながら、3行程度で書いてください。右端・左端・原点近くにあるジャンルの入れ替わりや、座標の目盛り（点の広がり）の変化が手がかりです。


（ここに3行程度で書く）


## 発展（任意）

### 多重対応分析 MCA：質的変数が3つ以上あるとき

今日の対応分析（CA）は、「ジャンル×年代」という **2つの質的変数のクロス表** を地図にしました。しかし実務では、アンケートの設問のように質的変数が3つも4つもあるのが普通です。

そのとき使うのが **多重対応分析（MCA）** です。各変数をダミー変数（0/1 の列）に展開した大きな表を作り、それに CA を当てる、と考えれば十分です。

数値データの散らばりを地図にするのが PCA、質的変数の結びつきを地図にするのが CA/MCA です。MCA は「PCA の質的変数版」と言えます。

地図の読み方は今日の CA と同じで、原点から同じ向きに遠いカテゴリどうしが結びついています。ダミー変数を並べた指示行列やバート表をどう作り、そこから何を分解しているのかという MCA 側の行列の扱いは、多重対応分析を実装したソフトウェアのドキュメント（R の FactoMineR パッケージの解説など）に譲ります。

ここでは第10回のペンギンデータの質的変数 3つ（`species`・`island`・`sex`）で試します。


In [ ]:
penguins = pd.read_csv(f'{DATA_DIR}/penguins.csv')
X_cat = penguins[['species', 'island', 'sex']].dropna()   # sex に欠損が 11 行あるので除く
print('行数:', len(X_cat))

mca = prince.MCA(n_components=2, random_state=42).fit(X_cat)
print(mca.eigenvalues_summary)


In [ ]:
# カテゴリ（列）の布置図。変数ごとに色を分ける
cat_coords = mca.column_coordinates(X_cat)
print(cat_coords.round(2))

colors = {'species': '#0066cc', 'island': '#e63946', 'sex': '#2a9d8f'}
fig, ax = plt.subplots(figsize=(8, 6))
for name, (x, y) in cat_coords.iloc[:, :2].iterrows():
    var, cat = name.split('__')            # prince 0.19 の列名形式（変数__カテゴリ）。例: 'species__Adelie' → ('species', 'Adelie')
    ax.scatter(x, y, c=colors[var], s=90, zorder=3)
    ax.annotate(f'{var}: {cat}', (x, y), xytext=(6, 4), textcoords='offset points', color=colors[var], fontsize=9)
ax.axhline(0, color='gray', lw=0.5, ls='--'); ax.axvline(0, color='gray', lw=0.5, ls='--')
ax.margins(0.15)
ax.set_xlabel('次元1'); ax.set_ylabel('次元2')
plt.tight_layout(); plt.show()


**読み方**　地図では、`Gentoo` と `Biscoe` が左に並び、`Chinstrap` と `Dream` が右下に、`Adelie` と `Torgersen` が上に並びます。原点から同じ向きに遠いので、この3組が強く結びついています。ただし Adelie は3島すべてにいるので、この組だけは「Torgersen にいれば Adelie」という片向きの結びつきです。実際に `pd.crosstab(X_cat['species'], X_cat['island'])` で確かめると、Gentoo は Biscoe 島だけ、Chinstrap は Dream 島だけ、Torgersen 島にいるのは Adelie だけです。

一方、`sex` の2点（Female・Male）は原点のすぐそば（座標の絶対値はどちらも 0.03 以下）にあります。どの種でも雌雄がほぼ半々なので、性別は種や島と結びついていない、と読めます。

寄与率は第1次元 36.2%、第2次元 28.9%、累積 65.1% でした。MCA の寄与率は CA より低めに出るのが普通なので、数値の大小より配置を読むのが実務的です。

試すなら、`heart.csv` の `sex`・`cp`・`thal`・`target` のような数値コードの列を `.astype(str)` で文字列にしてから渡してみてください。胸痛のタイプと心臓病の有無がどう結びつくかが地図になります。
